In [1]:
from google.colab import files
uploaded = files.upload()


Saving Document_QA_RAG_AgenticX.pdf to Document_QA_RAG_AgenticX.pdf


In [2]:
import os
print(os.listdir())

['.config', 'Document_QA_RAG_AgenticX.pdf', 'sample_data']


In [3]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 47.3 MB/s eta 0:00:00


In [4]:
from pypdf import PdfReader

pdf_path = list(uploaded.keys())[0]

reader = PdfReader(pdf_path)

print("Number of pages:", len(reader.pages))

Number of pages: 136


In [5]:
text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text + "\n"

print("Characters extracted:", len(text))
print(text[:2000])

Characters extracted: 381410
DIGITAL PROGRESS AND TRENDS REPORT 2025
STRENGTHENING  
AI FOUNDATIONS
Public Disclosure Authorized
Public Disclosure Authorized
Public Disclosure Authorized
Public Disclosure Authorized
Digital Progress and 
Trends Report 2025
This book, along with any associated content or subsequent updates,  
can be accessed at https://hdl.handle.net/10986/43822.
Scan to see all titles in this collection.
Digital Progress and 
Trends Report 2025
Strengthening AI Foundations
© 2025 International Bank for Reconstruction and Development / The World Bank
1818 H Street NW, Washington, DC 20433
Telephone: 202-473-1000; Internet: www.worldbank.org
Some rights reserved
1 2 3 4  28 27 26 25
This work is a product of the staff of The World Bank with external contributions. The findings, interpretations, and 
conclusions expressed in this work do not necessarily reflect the views of The World Bank, its Board of Executive Directors, 
or the governments they represent.
The World Ban

In [7]:
!pip install -q langchain-text-splitters

In [8]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter_1 = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks_1 = text_splitter_1.split_text(text)

print("Number of chunks:", len(chunks_1))
print("\nFirst chunk:\n")
print(chunks_1[0])

Number of chunks: 1

First chunk:

DIGITAL PROGRESS AND TRENDS REPORT 2025
STRENGTHENING  
AI FOUNDATIONS
Public Disclosure Authorized
Public Disclosure Authorized
Public Disclosure Authorized
Public Disclosure Authorized
Digital Progress and 
Trends Report 2025
This book, along with any associated content or subsequent updates,  
can be accessed at https://hdl.handle.net/10986/43822.
Scan to see all titles in this collection.
Digital Progress and 
Trends Report 2025
Strengthening AI Foundations
© 2025 International Bank for Reconstruction and Development / The World Bank
1818 H Street NW, Washington, DC 20433
Telephone: 202-473-1000; Internet: www.worldbank.org
Some rights reserved
1 2 3 4  28 27 26 25
This work is a product of the staff of The World Bank with external contributions. The findings, interpretations, and 
conclusions expressed in this work do not necessarily reflect the views of The World Bank, its Board of Executive Directors, 
or the governments they represent.
The Wor

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter_2 = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks_2 = text_splitter_2.split_text(text)

print("Number of chunks:", len(chunks_2))
print("\nFirst chunk:\n")
print(chunks_2[0])

Number of chunks: 429

First chunk:

DIGITAL PROGRESS AND TRENDS REPORT 2025
STRENGTHENING  
AI FOUNDATIONS
Public Disclosure Authorized
Public Disclosure Authorized
Public Disclosure Authorized
Public Disclosure Authorized
Digital Progress and 
Trends Report 2025
This book, along with any associated content or subsequent updates,  
can be accessed at https://hdl.handle.net/10986/43822.
Scan to see all titles in this collection.
Digital Progress and 
Trends Report 2025
Strengthening AI Foundations
© 2025 International Bank for Reconstruction and Development / The World Bank
1818 H Street NW, Washington, DC 20433
Telephone: 202-473-1000; Internet: www.worldbank.org
Some rights reserved
1 2 3 4  28 27 26 25
This work is a product of the staff of The World Bank with external contributions. The findings, interpretations, and 
conclusions expressed in this work do not necessarily reflect the views of The World Bank, its Board of Executive Directors, 
or the governments they represent.


In [10]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [11]:
embeddings = embedding_model.encode(
    chunks_2,
    show_progress_bar=True
)

print("Number of embeddings:", len(embeddings))
print("Embedding size:", embeddings.shape[1])

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Number of embeddings: 429
Embedding size: 384


In [12]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print("FAISS index created!")
print("Vectors stored:", index.ntotal)

FAISS index created!
Vectors stored: 429


In [13]:
def retrieve_chunks(question, k=3):
    question_embedding = embedding_model.encode([question])
    question_embedding = np.array(question_embedding).astype("float32")

    distances, indices = index.search(question_embedding, k)

    results = []

    for i, distance in zip(indices[0], distances[0]):
        results.append({
            "chunk": chunks_2[i],
            "distance": float(distance)
        })

    return results

In [14]:
question = "What is artificial intelligence?"

results = retrieve_chunks(question, k=3)

for i, result in enumerate(results, 1):
    print(f"\n--- Source {i} ---")
    print(result["chunk"])
    print("Distance:", result["distance"])


--- Source 1 ---
of performing tasks that typically require human intelligence. Although the concept dates to the 
1950s, in recent years, AI capabilities have improved significantly due to the availability of massive 
data, better training data, and more powerful computer hardware. 
AI is an umbrella term used for a set of loosely related technologies. No consensus exists on what 
is and is not AI (Narayanan and Kapoor 2024). As technological capabilities grow, tasks once 
exclusive to humans are now within the purview of machines. As a result, experts in the field often 
joke that AI is everything that computers cannot currently do. AI also has become a marketing 
buzzword, with companies rushing to label products as being AI based to ride the hype. Hence, it is 
challenging to provide a precise definition of AI or even of its types or subfields.
This publication defines AI broadly, encompassing subfields such as predictive AI based on machine
Distance: 0.7693901062011719

--- Sourc

In [16]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=150
)

print("Answer model loaded!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2

Answer model loaded!


In [17]:
def answer_question(question, k=3):
    results = retrieve_chunks(question, k)

    context = "\n\n".join([r["chunk"] for r in results])

    prompt = f"""
Answer the question using only the context below.

If the answer cannot be found in the context, say exactly:
I don't know

Context:
{context}

Question:
{question}

Answer:
"""

    response = generator(prompt)[0]["generated_text"]

    print("ANSWER:")
    print(response)

    print("\nSOURCES:")
    for i, result in enumerate(results, 1):
        print(f"\n--- Source {i} ---")
        print(result["chunk"][:500])

In [18]:
answer_question("What is artificial intelligence?")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (611 > 512). Running this sequence through the model will result in indexing errors
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER:

Answer the question using only the context below.

If the answer cannot be found in the context, say exactly:
I don't know

Context:
of performing tasks that typically require human intelligence. Although the concept dates to the 
1950s, in recent years, AI capabilities have improved significantly due to the availability of massive 
data, better training data, and more powerful computer hardware. 
AI is an umbrella term used for a set of loosely related technologies. No consensus exists on what 
is and is not AI (Narayanan and Kapoor 2024). As technological capabilities grow, tasks once 
exclusive to humans are now within the purview of machines. As a result, experts in the field often 
joke that AI is everything that computers cannot currently do. AI also has become a marketing 
buzzword, with companies rushing to label products as being AI based to ride the hype. Hence, it is 
challenging to provide a precise definition of AI or even of its types or subfields.
This publicati

In [19]:
def answer_question(question, k=3, threshold=1.5):
    results = retrieve_chunks(question, k)

    # Check whether the retrieved chunks are relevant
    best_distance = results[0]["distance"]

    if best_distance > threshold:
        print("ANSWER:")
        print("I don't know")
        print("\nNo sufficiently relevant information was found in the document.")
        return

    context = "\n\n".join([r["chunk"] for r in results])

    prompt = f"""
Answer the question using ONLY the context below.

If the answer is not present in the context, say exactly:
I don't know

Context:
{context}

Question:
{question}

Answer:
"""

    response = generator(prompt)[0]["generated_text"]

    print("ANSWER:")
    print(response)

    print("\nSOURCES:")
    for i, result in enumerate(results, 1):
        print(f"\n--- Source {i} ---")
        print(result["chunk"][:500])

In [20]:
answer_question("What is artificial intelligence?")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER:

Answer the question using ONLY the context below.

If the answer is not present in the context, say exactly:
I don't know

Context:
of performing tasks that typically require human intelligence. Although the concept dates to the 
1950s, in recent years, AI capabilities have improved significantly due to the availability of massive 
data, better training data, and more powerful computer hardware. 
AI is an umbrella term used for a set of loosely related technologies. No consensus exists on what 
is and is not AI (Narayanan and Kapoor 2024). As technological capabilities grow, tasks once 
exclusive to humans are now within the purview of machines. As a result, experts in the field often 
joke that AI is everything that computers cannot currently do. AI also has become a marketing 
buzzword, with companies rushing to label products as being AI based to ride the hype. Hence, it is 
challenging to provide a precise definition of AI or even of its types or subfields.
This publicatio

In [21]:
answer_question("What is the recipe for making pizza?")

ANSWER:
I don't know

No sufficiently relevant information was found in the document.


In [22]:
def answer_question_with_sources(question, k=3, threshold=1.5):
    results = retrieve_chunks(question, k)

    best_distance = results[0]["distance"]

    if best_distance > threshold:
        print("ANSWER:")
        print("I don't know")
        print("\nNo sufficiently relevant information was found in the document.")
        return

    context = "\n\n".join([r["chunk"] for r in results])

    prompt = f"""
Answer the question using ONLY the context below.
If the answer is not present in the context, say exactly: I don't know

Context:
{context}

Question:
{question}

Answer:
"""

    response = generator(prompt)[0]["generated_text"]

    print("=" * 60)
    print("ANSWER")
    print("=" * 60)
    print(response)

    print("\n" + "=" * 60)
    print("CITATIONS / SOURCE PASSAGES")
    print("=" * 60)

    for i, result in enumerate(results, 1):
        print(f"\n[Source {i}]")
        print(result["chunk"][:700])

In [23]:
answer_question_with_sources(
    "What are some important developments in artificial intelligence?"
)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER

Answer the question using ONLY the context below.
If the answer is not present in the context, say exactly: I don't know

Context:
AI development tools
GitHub Copilot (Microsoft), 
Hugging Face, PyTorch (Meta), 
TensorFlow (Google) 
Traditional AI 
algorithms and models
Alphabet, Apple, Meta, 
TikTok, Tesla
Generative AI models
Alphabet, Anthropic, DeepSeek,  
Mistral AI, Meta, Microsoft,
OpenAI 
AI application
AI application software
Adobe, Grammarly, Harvey, Microsoft, OpenAI, 
Stability AI, Jasper.ai, Soul Machines
AI-powered devices and equipment
Boston Dynamics, Intuitive Surgical, iRobot, 
Tesla, Samsung, Waymo, Xiaomi 
Source: Original figure for this publication.
Note: AI = artificial intelligence.
4  DIGITAL PROGRESS AND TRENDS REPORT 2025  
Training data are yet another critical part of this ecosystem, which can be gathered from various 
sources such as digital devices, sensors, digital platforms, business transactions, and public data sets. 
These data are then aggre

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

page_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

page_chunks = []
page_numbers = []

for page_num, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text()

    if page_text:
        chunks = page_splitter.split_text(page_text)

        for chunk in chunks:
            page_chunks.append(chunk)
            page_numbers.append(page_num)

print("Total chunks:", len(page_chunks))
print("Total page references:", len(page_numbers))

Total chunks: 488
Total page references: 488


In [25]:
page_embeddings = embedding_model.encode(
    page_chunks,
    show_progress_bar=True
)

page_embeddings = np.array(page_embeddings).astype("float32")

page_index = faiss.IndexFlatL2(page_embeddings.shape[1])
page_index.add(page_embeddings)

print("FAISS index created!")
print("Vectors stored:", page_index.ntotal)

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

FAISS index created!
Vectors stored: 488


In [27]:
def retrieve_page_chunks(question, k=3):
    question_embedding = embedding_model.encode([question])
    question_embedding = np.array(question_embedding).astype("float32")

    distances, indices = page_index.search(question_embedding, k)

    results = []

    for i, distance in zip(indices[0], distances[0]):
        results.append({
            "chunk": page_chunks[i],
            "distance": float(distance),
            "page": page_numbers[i]
        })

    return results

results = retrieve_page_chunks(
    "What is artificial intelligence?",
    k=3
)

for i, result in enumerate(results, 1):
    print(f"\n--- Source {i} | Page {result['page']} ---")
    print(result["chunk"][:500])


--- Source 1 | Page 24 ---
use AI for lesson planning, nurses for patient tracking, contact center agents for coaching, 
and farmers for agronomic advice. In these roles, AI acts more as a co-worker or coach—
enhancing human capacity—than a replacement.
 ◦ However, the scale, equity, and sustainability of benefits hinge on the 4Cs.
AI definitions, ecosystem, and significant trends
Artificial intelligence  (AI) is a branch of computer science dedicated to creating systems capable 
of performing tasks that typically require

--- Source 2 | Page 28 ---
a degree of autonomy, making decisions based on the data they have learned from and tackling 
more complex cognitive problems. This autonomy in optimization and decision-making allows 
AI to either substitute for or complement human decision-making and judgment.
2. Data dependency. AI relies heavily on large data sets for training and operation. This contrasts 
with technologies such as the steam engine, electricity, internal combustion en

In [28]:
def answer_with_page_citations(question, k=3, threshold=1.5):
    results = retrieve_page_chunks(question, k)

    # Check relevance
    if not results or results[0]["distance"] > threshold:
        print("ANSWER:")
        print("I don't know")
        print("\nNo sufficiently relevant information was found in the document.")
        return

    context = "\n\n".join(
        [f"[Page {r['page']}]\n{r['chunk']}" for r in results]
    )

    prompt = f"""
Answer the question using ONLY the context below.

If the answer is not present in the context, say exactly:
I don't know

Context:
{context}

Question:
{question}

Answer:
"""

    response = generator(prompt)[0]["generated_text"]

    print("=" * 60)
    print("ANSWER")
    print("=" * 60)
    print(response)

    print("\n" + "=" * 60)
    print("CITATIONS")
    print("=" * 60)

    for i, result in enumerate(results, 1):
        print(f"\n[Source {i} | Page {result['page']}]")
        print(result["chunk"][:700])

In [29]:
answer_with_page_citations(
    "What is artificial intelligence?"
)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER

Answer the question using ONLY the context below.

If the answer is not present in the context, say exactly:
I don't know

Context:
[Page 24]
use AI for lesson planning, nurses for patient tracking, contact center agents for coaching, 
and farmers for agronomic advice. In these roles, AI acts more as a co-worker or coach—
enhancing human capacity—than a replacement.
 ◦ However, the scale, equity, and sustainability of benefits hinge on the 4Cs.
AI definitions, ecosystem, and significant trends
Artificial intelligence  (AI) is a branch of computer science dedicated to creating systems capable 
of performing tasks that typically require human intelligence. Although the concept dates to the 
1950s, in recent years, AI capabilities have improved significantly due to the availability of massive 
data, better training data, and more powerful computer hardware. 
AI is an umbrella term used for a set of loosely related technologies. No consensus exists on what 
is and is not AI (Naraya

In [30]:
answer_with_page_citations(
    "What are the main applications of artificial intelligence?"
)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER

Answer the question using ONLY the context below.

If the answer is not present in the context, say exactly:
I don't know

Context:
[Page 26]
watch a video, listen to the audio, read the subtitles, and generate a comprehensive summary of 
what occurred. This allows for a much richer, more contextual understanding of the world and 
enables more fluid and intuitive interactions with technology, moving it closer to how humans 
perceive reality. 
Beyond better perception, the field is intensely focused on advanced AI reasoning. Although current 
models are excellent at pattern recognition, they can struggle with multistep problems that require 
logical deduction, strategic planning, or creative problem-solving. The next frontier is to develop AI 
that can not only retrieve information but can also reason with it to analyze a complex problem, 
break it into smaller steps, form a coherent plan, and execute it. This capability is essential for 
tackling grand challenges in science, me

In [31]:
answer_with_page_citations(
    "What are some challenges associated with artificial intelligence?"
)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER

Answer the question using ONLY the context below.

If the answer is not present in the context, say exactly:
I don't know

Context:
[Page 45]
2024. https://www.wsj.com/tech/ai/yann-lecun-ai-meta-aa59e2f5.
Mirzadeh, I., K. Alizadeh, H. Shahrokhi, O. Tuzel, S. Bengio, and M. Farajtabar. 2024. “GSM-Symbolic: 
Understanding the Limitations of Mathematical Reasoning in Large Language Models.” Preprint, last 
revised August 27, 2025. https://arxiv.org/abs/2410.05229.
Narayanan, A., and S. Kapoor. 2024. AI Snake Oil: What Artificial Intelligence Can Do, What It Can’t, 
and How to Tell the Difference. Princeton, NJ: Princeton University Press.
Narayanan, A., and S. Kapoor. 2025. AI as Normal Technology. New York: Knight First Amendment 
Institute, Columbia University, New York. https://knightcolumbia.org/content/ai-as-normal -technology.
Noy, S., and W. Zhang. 2023. “Experimental Evidence on the Productivity Effects of Generative Artificial 
Intelligence.” Science 381 (6654): 187–92.
O

In [32]:
answer_with_page_citations(
    "What is the capital city of Japan?"
)

ANSWER:
I don't know

No sufficiently relevant information was found in the document.


In [33]:
print("CHUNKING COMPARISON")
print("=" * 50)

print("\nStrategy 1: Fixed Character Chunking")
print("Number of chunks:", len(chunks_1))
print("Average chunk length:",
      sum(len(c) for c in chunks_1) / len(chunks_1))

print("\nStrategy 2: Recursive Character Chunking")
print("Number of chunks:", len(chunks_2))
print("Average chunk length:",
      sum(len(c) for c in chunks_2) / len(chunks_2))

CHUNKING COMPARISON

Strategy 1: Fixed Character Chunking
Number of chunks: 1
Average chunk length: 381409.0

Strategy 2: Recursive Character Chunking
Number of chunks: 429
Average chunk length: 954.3799533799533


In [34]:
def retrieve_with_index(question, chunks, index, k=3):
    question_embedding = embedding_model.encode([question])
    question_embedding = np.array(question_embedding).astype("float32")

    distances, indices = index.search(question_embedding, k)

    return [
        {
            "chunk": chunks[i],
            "distance": float(distance)
        }
        for i, distance in zip(indices[0], distances[0])
    ]

In [35]:
embeddings_1 = embedding_model.encode(
    chunks_1,
    show_progress_bar=True
)

embeddings_1 = np.array(embeddings_1).astype("float32")

index_1 = faiss.IndexFlatL2(embeddings_1.shape[1])
index_1.add(embeddings_1)

print("Strategy 1 index:", index_1.ntotal)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Strategy 1 index: 1


In [36]:
questions = [
    "What is artificial intelligence?",
    "What are the challenges of artificial intelligence?",
    "How is AI being used in different sectors?"
]

for question in questions:
    results_1 = retrieve_with_index(
        question, chunks_1, index_1
    )

    results_2 = retrieve_with_index(
        question, chunks_2, index,
    )

    print("\n" + "=" * 70)
    print("QUESTION:", question)

    print("\nStrategy 1 - Best distance:",
          results_1[0]["distance"])

    print("Strategy 2 - Best distance:",
          results_2[0]["distance"])


QUESTION: What is artificial intelligence?

Strategy 1 - Best distance: 1.4968130588531494
Strategy 2 - Best distance: 0.7693901062011719

QUESTION: What are the challenges of artificial intelligence?

Strategy 1 - Best distance: 1.459427833557129
Strategy 2 - Best distance: 0.8886390924453735

QUESTION: How is AI being used in different sectors?

Strategy 1 - Best distance: 1.3863714933395386
Strategy 2 - Best distance: 0.7529107332229614


In [37]:
def compare_retrieval(question, k=3):
    results_1 = retrieve_with_index(
        question, chunks_1, index_1, k
    )

    results_2 = retrieve_with_index(
        question, chunks_2, index, k
    )

    print("=" * 70)
    print("QUESTION:", question)

    print("\n--- STRATEGY 1: FIXED-SIZE ---")
    for i, r in enumerate(results_1, 1):
        print(f"\nSource {i} | Distance: {r['distance']:.4f}")
        print(r["chunk"][:400])

    print("\n--- STRATEGY 2: RECURSIVE ---")
    for i, r in enumerate(results_2, 1):
        print(f"\nSource {i} | Distance: {r['distance']:.4f}")
        print(r["chunk"][:400])

In [38]:
compare_retrieval(
    "What are the main applications of artificial intelligence?"
)

QUESTION: What are the main applications of artificial intelligence?

--- STRATEGY 1: FIXED-SIZE ---

Source 1 | Distance: 1.5462
DIGITAL PROGRESS AND TRENDS REPORT 2025
STRENGTHENING  
AI FOUNDATIONS
Public Disclosure Authorized
Public Disclosure Authorized
Public Disclosure Authorized
Public Disclosure Authorized
Digital Progress and 
Trends Report 2025
This book, along with any associated content or subsequent updates,  
can be accessed at https://hdl.handle.net/10986/43822.
Scan to see all titles in this collection.
Digi

Source 2 | Distance: 340282346638528859811704183484516925440.0000
DIGITAL PROGRESS AND TRENDS REPORT 2025
STRENGTHENING  
AI FOUNDATIONS
Public Disclosure Authorized
Public Disclosure Authorized
Public Disclosure Authorized
Public Disclosure Authorized
Digital Progress and 
Trends Report 2025
This book, along with any associated content or subsequent updates,  
can be accessed at https://hdl.handle.net/10986/43822.
Scan to see all titles in this collection.
Digi



In [39]:
test_questions = [
    "What is artificial intelligence?",
    "What are the main applications of artificial intelligence?",
    "What are the challenges of artificial intelligence?",
    "How is AI used in different sectors?",
    "What are the foundations needed for AI?",
    "What factors affect AI adoption?"
]

for question in test_questions:
    print("\n" + "=" * 70)
    print("QUESTION:", question)

    r1 = retrieve_with_index(question, chunks_1, index_1, k=1)[0]
    r2 = retrieve_with_index(question, chunks_2, index, k=1)[0]

    print(f"Fixed-size distance: {r1['distance']:.4f}")
    print(f"Recursive distance:  {r2['distance']:.4f}")


QUESTION: What is artificial intelligence?
Fixed-size distance: 1.4968
Recursive distance:  0.7694

QUESTION: What are the main applications of artificial intelligence?
Fixed-size distance: 1.5462
Recursive distance:  0.9058

QUESTION: What are the challenges of artificial intelligence?
Fixed-size distance: 1.4594
Recursive distance:  0.8886

QUESTION: How is AI used in different sectors?
Fixed-size distance: 1.4107
Recursive distance:  0.7300

QUESTION: What are the foundations needed for AI?
Fixed-size distance: 1.2131
Recursive distance:  0.7497

QUESTION: What factors affect AI adoption?
Fixed-size distance: 1.4107
Recursive distance:  0.7528


In [40]:
import pandas as pd

comparison_results = []

for question in test_questions:
    r1 = retrieve_with_index(question, chunks_1, index_1, k=1)[0]
    r2 = retrieve_with_index(question, chunks_2, index, k=1)[0]

    comparison_results.append({
        "Question": question,
        "Fixed-size Distance": round(r1["distance"], 4),
        "Recursive Distance": round(r2["distance"], 4)
    })

comparison_df = pd.DataFrame(comparison_results)

comparison_df

,Question,Fixed-size Distance,Recursive Distance
0,What is artificial intelligence?,1.4968,0.7694
1,What are the main applications of artificial i...,1.5462,0.9058
2,What are the challenges of artificial intellig...,1.4594,0.8886
3,How is AI used in different sectors?,1.4107,0.7300
4,What are the foundations needed for AI?,1.2131,0.7497
5,What factors affect AI adoption?,1.4107,0.7528


In [41]:
print("Average Fixed-size Distance:",
      comparison_df["Fixed-size Distance"].mean())

print("Average Recursive Distance:",
      comparison_df["Recursive Distance"].mean())

Average Fixed-size Distance: 1.4228166666666666
Average Recursive Distance: 0.7993833333333332


In [42]:
evaluation_questions = [
    "What is artificial intelligence?",
    "What are the main applications of artificial intelligence?",
    "What are the challenges of artificial intelligence?",
    "How is AI used in different sectors?",
    "What are the foundations needed for AI?",
    "What factors affect AI adoption?"
]

for question in evaluation_questions:
    print("\n" + "=" * 80)
    print("QUESTION:", question)

    results = retrieve_page_chunks(question, k=2)

    for i, r in enumerate(results, 1):
        print(f"\nSOURCE {i} | PDF PAGE {r['page']}")
        print("-" * 60)
        print(r["chunk"][:1000])


QUESTION: What is artificial intelligence?

SOURCE 1 | PDF PAGE 24
------------------------------------------------------------
use AI for lesson planning, nurses for patient tracking, contact center agents for coaching, 
and farmers for agronomic advice. In these roles, AI acts more as a co-worker or coach—
enhancing human capacity—than a replacement.
 ◦ However, the scale, equity, and sustainability of benefits hinge on the 4Cs.
AI definitions, ecosystem, and significant trends
Artificial intelligence  (AI) is a branch of computer science dedicated to creating systems capable 
of performing tasks that typically require human intelligence. Although the concept dates to the 
1950s, in recent years, AI capabilities have improved significantly due to the availability of massive 
data, better training data, and more powerful computer hardware. 
AI is an umbrella term used for a set of loosely related technologies. No consensus exists on what 
is and is not AI (Narayanan and Kapoor 2024).

In [43]:
evaluation_results = []

for question in evaluation_questions:
    r1 = retrieve_with_index(question, chunks_1, index_1, k=1)[0]
    r2 = retrieve_with_index(question, chunks_2, index, k=1)[0]

    evaluation_results.append({
        "Question": question,
        "Fixed-size Distance": round(r1["distance"], 4),
        "Recursive Distance": round(r2["distance"], 4),
        "Lower Distance": (
            "Fixed-size"
            if r1["distance"] < r2["distance"]
            else "Recursive"
        )
    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,Question,Fixed-size Distance,Recursive Distance,Lower Distance
0,What is artificial intelligence?,1.4968,0.7694,Recursive
1,What are the main applications of artificial i...,1.5462,0.9058,Recursive
2,What are the challenges of artificial intellig...,1.4594,0.8886,Recursive
3,How is AI used in different sectors?,1.4107,0.7300,Recursive
4,What are the foundations needed for AI?,1.2131,0.7497,Recursive
5,What factors affect AI adoption?,1.4107,0.7528,Recursive
